# Problem 1
Let $n_0$ and $n_1$ be the number of tests where $H_0$ and $H_1$ holds, respectively. The number of false positives is then $\text{FP}= \alpha n_0$ and the number of false negatives is $\text{FN}=(1-\beta)n_1$, where $\beta$ is the power. From this it follows that the number of true negatives is $\text{TN}=(1-\alpha)n_0$ and the number of true positives is $\text{TP}=\beta h_1$. Substituting $\alpha=0.05$, we find
| case | $n_0$ | $n_1$ | $\beta$ | TP | FP | TN | FN |
| ---- | ---- | ---- | ---- | ---- | ---- | ---- | ---- |
| 1 | 100 | 100 | 0.8 | 80 | 5 | 95 | 20 |
| 2 | 100 | 100 | 0.2 | 20 | 5 | 95 | 80 |
| 3 | 900 | 100 | 0.8 | 80 | 45 | 855 | 20 |
| 4 | 900 | 100 | 0.2 | 20 | 45 | 855 | 80 |

TBD: Error rates


# Problem 2

We are testing if the difference in mean accuracy is different from zero, i.e. $H_0: \delta_{ij} = \mu_i - \mu_j = 0$ vs. $H_1 : \delta_{ij}\neq 0$, for each pair of algorithms $(i,j)$. The estimator is then $\hat{\delta}_{ij}= \hat{\mu}_i - \hat{\mu}_j$. For independent samples $\text{Var}(\hat{\delta}_{ij})= \frac{\hat{\sigma}_i^2}{n} + \frac{\hat{\sigma}_j^2}{n}$ and $\hat{\text{se}}(\hat{\delta}_{ij}) = \sqrt{\text{Var}(\hat{\delta}_{ij})}$. Since $\hat{\sigma}_i=2$ for all $i=1,2,3$ and $n=8$, we find that $\hat{\text{se}}(\hat{\delta}_{ij})=\sqrt{2\cdot\frac{4}{8}}=1$. Thus the Wald test statistic is
$$
W_{ij} = \frac{\hat{\delta}_{ij}}{\hat{\text{se}}(\hat{\delta}_{ij})} = \hat{\delta}_{ij},
$$
and the $p$-value is obtained as
$$
p_{ij} = \mathbb{P}(|Z| > |W_{ij}|) = 2 \Phi(-|W_{ij}|)= 2(1-\Phi(|W_{ij}|)).
$$

So we find
| $(i,j)$ | $\|W_{ij}\|$ | $p_{ij}$ |
| ---- | ---- | ---- |
| $(1,2)$ | $1$ | $2(1-\Phi(1))=0.31732$ |
| $(1,3)$ | $2.5$ | $2(1-\Phi(2.5))=0.01242$ |
| $(2,3)$ | $1.5$ | $2(1-\Phi(1.5))=0.13362$ |

At 5% significance level only the difference between algorithm 1 and 3 is significant. Therefore it can't be justified that algorithm 1 is the best out of all three.

# Problem 3

We are testing $H_0 : p_\text{placebo} = p_\text{drug}$ vs. $H_1 : p_\text{drug} < p_\text{placebo}$ for each drug. The placebo estimate is $\hat{p}_p=n_\text{nausea}/n_\text{patients}=45/80$ and likewise for each drug. The standard error for a proportion is $\hat{\text{se}}=\sqrt{\frac{p(1-p)}{n}}$ where instead of $p$ we use the plug-in estimate $\hat{p}$. For a drug $D$ the Wald test statistic is then
$$
W = \frac{\hat{p}_D - \hat{p}_p}{\sqrt{\frac{\hat{p}_D(1-\hat{p}_D)}{n_D} + \frac{\hat{p}_p(1-\hat{p}_p)}{n_p}}}.
$$
Since we are doing one-sided test, the $p$-value is $p=\Phi(W)$.

| Drug | $\hat{p}_D$ | $W$ | $p$ |
| ---- | ---- | ---- | ---- |
| Chlorpromazine | $26/75$ | $-2.764$ | $0.003$ |
| Dimenhydrinate | $52/85$ | $0.643$ | $0.740$ |
| Pentobarbital (100 mg) | $35/67$ | $-0.486$ | $0.313$ |
| Pentobarbital (150 mg) | $36/84$ | $-1.730$ | $0.042$ |

Based on the one-sided Wald test, the incidence of nausea is lower than the placebo only for Chlorpromazine and Pentobarbital (150 mg) at the 5% level.

In [8]:
import numpy as np
from scipy import stats

drugs = ["Chlorpromazine", "Dimenhydrinate", "Pentobarbital (100 mg)", "Pentobarbital (150 mg)"]
n_n = [26, 52, 35, 36] # incidence of nausea
n_p = [75, 85, 67, 84] # n of patients
p_placebo = 45/80
p_vals = {}

for drug, n_n, n_p in zip(drugs, n_n, n_p):
    p_drug = n_n/n_p
    se = np.sqrt(p_placebo*(1-p_placebo)/80 + p_drug*(1-p_drug)/n_p)
    W = (p_drug -  p_placebo) / se
    p_val = stats.norm.cdf(W)
    p_vals[drug] = p_val
    print(f"Drug = {drug}, W = {W:.3f}, p = {p_val:.5f}")


Drug = Chlorpromazine, W = -2.764, p = 0.00285
Drug = Dimenhydrinate, W = 0.643, p = 0.73988
Drug = Pentobarbital (100 mg), W = -0.486, p = 0.31333
Drug = Pentobarbital (150 mg), W = -1.730, p = 0.04180


# Problem 4

We have 4 tests in total. Only Chlorpromazine is significant at 5% level after adjusting for multiple testing.


In [17]:
n_tests = 4
for drug, p_val in p_vals.items():
    print(f"{drug}, Bonferroni p-val = {np.min([1,p_val*n_tests]):.4f}")

p_vals_sorted = dict(sorted(p_vals.items(), key=lambda item: item[1]))

i = 1
for drug, p_val in p_vals_sorted.items():
    print(f"{drug}, BH p-val = {np.min([1,p_val*n_tests/i]):.4f}")
    i += 1

Chlorpromazine, Bonferroni p-val = 0.0114
Dimenhydrinate, Bonferroni p-val = 1.0000
Pentobarbital (100 mg), Bonferroni p-val = 1.0000
Pentobarbital (150 mg), Bonferroni p-val = 0.1672
Chlorpromazine, BH p-val = 0.0114
Pentobarbital (150 mg), BH p-val = 0.0836
Pentobarbital (100 mg), BH p-val = 0.4178
Dimenhydrinate, BH p-val = 0.7399


# Problem 5
## 1.
The MLE of $p$ is $\hat{p}=(\hat{p}_1,\dots ,\hat{p}_6)$. Each component $\hat{p}_j$ is computed as
$$
\hat{p}_j = \frac{1}{n} \sum_{i=1}^n I(X_i = j),
$$
for $j=1,\dots, 6$. Inserting $n=20$ we thus find
$$
\hat{p} = (0.2, 0.25, 0.2, 0, 0.1, 0.25).
$$

## 2.
Since the throws are independent events, the predictive likelihood is $\hat{p}_1\hat{p}_5\hat{p}_4 = 0.2 \cdot 0.1 \cdot 0=0$.

## 3.
We add $\alpha$ to each of the original counts and then divide by the total sum to get the posterior distribution.

In [19]:
counts = np.array([4, 5, 4, 0, 2, 5])
for alpha in [0.1, 1, 10]:
    new_counts = alpha + counts
    posterior = new_counts / new_counts.sum()
    print(f"Posterior distribution for alpha = {alpha}:")
    print(posterior)

Posterior distribution for alpha = 0.1:
[0.19902913 0.24757282 0.19902913 0.00485437 0.10194175 0.24757282]
Posterior distribution for alpha = 1:
[0.19230769 0.23076923 0.19230769 0.03846154 0.11538462 0.23076923]
Posterior distribution for alpha = 10:
[0.175  0.1875 0.175  0.125  0.15   0.1875]
